In [ ]:
from pathlib import Path
import pandas as pd
from scapy.all import *
from scapy.error import Scapy_Exception

root = Path.cwd()

pcap_files = sorted(root.rglob("*.pcap")) + sorted(root.rglob("*.pcapng"))
pcap_files = [p for p in pcap_files if p.is_file()]

out_root = root / "csv_output"
out_root.mkdir(exist_ok=True)


def load_capture(path):
    path = str(path)

    with open(path, "rb") as f:
        magic = f.read(4)

    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)

    return PcapReader(path)


def packet_to_dict(pkt, pcap_path):

    row = {}

    row["pcap_file"] = str(pcap_path.relative_to(root))
    row["timestamp"] = float(pkt.time)
    row["packet_length"] = len(pkt)

    # ---------------- Ethernet ----------------

    if Ether in pkt:
        row["src_mac"] = pkt[Ether].src
        row["dst_mac"] = pkt[Ether].dst
        row["eth_type"] = pkt[Ether].type
    else:
        row["src_mac"] = None
        row["dst_mac"] = None
        row["eth_type"] = None

    # ---------------- IP ----------------

    if IP in pkt:

        ip = pkt[IP]

        row["src_ip"] = ip.src
        row["dst_ip"] = ip.dst
        row["ip_version"] = ip.version
        row["ttl"] = ip.ttl
        row["ip_length"] = ip.len
        row["tos"] = ip.tos
        row["id"] = ip.id
        row["flags_ip"] = int(ip.flags)
        row["fragment"] = ip.frag
        row["protocol"] = ip.proto

    else:

        row["src_ip"] = None
        row["dst_ip"] = None
        row["ip_version"] = None
        row["ttl"] = None
        row["ip_length"] = None
        row["tos"] = None
        row["id"] = None
        row["flags_ip"] = None
        row["fragment"] = None
        row["protocol"] = None

    # ---------------- TCP ----------------

    if TCP in pkt:

        tcp = pkt[TCP]

        row["src_port"] = tcp.sport
        row["dst_port"] = tcp.dport
        row["seq"] = tcp.seq
        row["ack"] = tcp.ack
        row["window"] = tcp.window

        row["syn"] = int(tcp.flags.S)
        row["ack_flag"] = int(tcp.flags.A)
        row["fin"] = int(tcp.flags.F)
        row["rst"] = int(tcp.flags.R)
        row["psh"] = int(tcp.flags.P)
        row["urg"] = int(tcp.flags.U)

    else:

        row["src_port"] = None
        row["dst_port"] = None
        row["seq"] = None
        row["ack"] = None
        row["window"] = None
        row["syn"] = 0
        row["ack_flag"] = 0
        row["fin"] = 0
        row["rst"] = 0
        row["psh"] = 0
        row["urg"] = 0

    # ---------------- UDP ----------------

    if UDP in pkt:

        udp = pkt[UDP]

        row["udp_length"] = udp.len

    else:

        row["udp_length"] = None

    # ---------------- ICMP ----------------

    if ICMP in pkt:

        icmp = pkt[ICMP]

        row["icmp_type"] = icmp.type
        row["icmp_code"] = icmp.code

    else:

        row["icmp_type"] = None
        row["icmp_code"] = None

    # Payload size

    row["payload_size"] = len(bytes(pkt.payload))

    return row


for pcap_path in pcap_files:

    print(f"Processing {pcap_path.name}")

    try:

        packets = load_capture(pcap_path)

        rows = []

        for pkt in packets:
            rows.append(packet_to_dict(pkt, pcap_path))

        df = pd.DataFrame(rows)

        csv_path = out_root / pcap_path.relative_to(root).with_suffix(".csv")
        csv_path.parent.mkdir(parents=True, exist_ok=True)

        df.to_csv(csv_path, index=False)

        print(f"Saved {len(df)} packets")

    except Scapy_Exception as ex:

        print(f"Skipping {pcap_path}: {ex}")

In [3]:
# Balanced Version - Fast & Model-Ready
from pathlib import Path
import pandas as pd
from scapy.all import *

root = Path("DDOS/DDOS/Network_Layer")

pcap_files = sorted(root.rglob("*.pcap"))[:10]  # Process first 10 files

out_root = root / "csv_output"
out_root.mkdir(exist_ok=True)

def load_capture(path):
    path = str(path)
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)
    return PcapReader(path)

def extract_packet(pkt, pcap_path):
    try:
        row = {
            "pcap_file": str(pcap_path.name),
            "timestamp": float(pkt.time),
            "packet_length": len(pkt),
        }
        
        # IP Layer
        if IP in pkt:
            row["src_ip"] = pkt[IP].src
            row["dst_ip"] = pkt[IP].dst
            row["protocol"] = pkt[IP].proto
            row["ttl"] = pkt[IP].ttl
            row["tos"] = pkt[IP].tos
        else:
            row["src_ip"] = row["dst_ip"] = row["protocol"] = row["ttl"] = row["tos"] = None
        
        # TCP Layer
        if TCP in pkt:
            tcp = pkt[TCP]
            row["src_port"] = tcp.sport
            row["dst_port"] = tcp.dport
            row["tcp_flags"] = int(tcp.flags)
            row["seq"] = tcp.seq
            row["ack"] = tcp.ack
            row["window"] = tcp.window
        else:
            row["src_port"] = row["dst_port"] = row["tcp_flags"] = None
            row["seq"] = row["ack"] = row["window"] = None
        
        # UDP Layer
        if UDP in pkt:
            row["udp_len"] = pkt[UDP].len
        else:
            row["udp_len"] = None
        
        # ICMP Layer (important for DDoS detection)
        if ICMP in pkt:
            row["icmp_type"] = pkt[ICMP].type
            row["icmp_code"] = pkt[ICMP].code
        else:
            row["icmp_type"] = row["icmp_code"] = None
        
        return row
    except:
        return None

for pcap_path in pcap_files:
    print(f"Processing {pcap_path.name}...", end=" ")
    
    try:
        packets = load_capture(pcap_path)
        rows = [r for pkt in packets for r in [extract_packet(pkt, pcap_path)] if r is not None]
        
        if rows:
            df = pd.DataFrame(rows)
            csv_path = out_root / f"{pcap_path.stem}.csv"
            df.to_csv(csv_path, index=False)
            print(f"✓ {len(df)} packets")
        else:
            print("✗ No packets")
    except Exception as e:
        print(f"✗ {str(e)[:40]}")

Processing ddos_ack_hping3.pcap... ✓ 29834839 packets
Processing ddos_ack_nping.pcap... ✗ Unable to allocate 90.4 MiB for an array
Processing ddos_icmp_hping3.pcap... ✓ 409452 packets
Processing ddos_icmp_nping.pcap... ✓ 11439345 packets
Processing ddos_slowloris.pcap... ✓ 3424465 packets
Processing ddos_syn_hping3.pcap... ✓ 14843551 packets
Processing ddos_syn_nping.pcap... ✓ 4329827 packets
Processing ddos_udp_hping3.pcap... ✓ 2007579 packets
Processing ddos_udp_nping.pcap... ✓ 6750037 packets


In [4]:
# Balanced Version - Fast & Model-Ready
from pathlib import Path
import pandas as pd
from scapy.all import *

root = Path("DOS/DOS/Network_Layer")

pcap_files = sorted(root.rglob("*.pcap"))[:10]  # Process first 10 files

out_root = root / "csv_output"
out_root.mkdir(exist_ok=True)

def load_capture(path):
    path = str(path)
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)
    return PcapReader(path)

def extract_packet(pkt, pcap_path):
    try:
        row = {
            "pcap_file": str(pcap_path.name),
            "timestamp": float(pkt.time),
            "packet_length": len(pkt),
        }
        
        # IP Layer
        if IP in pkt:
            row["src_ip"] = pkt[IP].src
            row["dst_ip"] = pkt[IP].dst
            row["protocol"] = pkt[IP].proto
            row["ttl"] = pkt[IP].ttl
            row["tos"] = pkt[IP].tos
        else:
            row["src_ip"] = row["dst_ip"] = row["protocol"] = row["ttl"] = row["tos"] = None
        
        # TCP Layer
        if TCP in pkt:
            tcp = pkt[TCP]
            row["src_port"] = tcp.sport
            row["dst_port"] = tcp.dport
            row["tcp_flags"] = int(tcp.flags)
            row["seq"] = tcp.seq
            row["ack"] = tcp.ack
            row["window"] = tcp.window
        else:
            row["src_port"] = row["dst_port"] = row["tcp_flags"] = None
            row["seq"] = row["ack"] = row["window"] = None
        
        # UDP Layer
        if UDP in pkt:
            row["udp_len"] = pkt[UDP].len
        else:
            row["udp_len"] = None
        
        # ICMP Layer (important for DDoS detection)
        if ICMP in pkt:
            row["icmp_type"] = pkt[ICMP].type
            row["icmp_code"] = pkt[ICMP].code
        else:
            row["icmp_type"] = row["icmp_code"] = None
        
        return row
    except:
        return None

for pcap_path in pcap_files:
    print(f"Processing {pcap_path.name}...", end=" ")
    
    try:
        packets = load_capture(pcap_path)
        rows = [r for pkt in packets for r in [extract_packet(pkt, pcap_path)] if r is not None]
        
        if rows:
            df = pd.DataFrame(rows)
            csv_path = out_root / f"{pcap_path.stem}.csv"
            df.to_csv(csv_path, index=False)
            print(f"✓ {len(df)} packets")
        else:
            print("✗ No packets")
    except Exception as e:
        print(f"✗ {str(e)[:40]}")

Processing icmp_hping3.pcap... ✓ 784241 packets
Processing icmp_nping.pcap... ✓ 5657405 packets
Processing slowloris.pcap... ✓ 2726636 packets
Processing slowloris2.pcap... ✓ 3051120 packets
Processing syn_hping3.pcap... ✗ Unable to allocate 2.92 GiB for an array
Processing syn_nping.pcap... ✓ 2503788 packets
Processing tcp.pcap... ✓ 502864 packets
Processing tcp_ack_hping3.pcap... ✓ 13186324 packets
Processing tcp_nping.pcap... ✓ 5127143 packets
Processing ucp_hping3_2.pcap... ✓ 9169658 packets


In [1]:
# Merge all CSVs from specific folders with proper labels (handles varying columns)
from pathlib import Path
import pandas as pd

root = Path.cwd()
csv_root = root / "csv_output"

# Specific folders to scan
target_paths = [
    csv_root / "Benign/Benign/Network_data",
    csv_root / "BruteForce/BruteForce/Higher_Layer",
    csv_root / "DDOS/DDOS",
    csv_root / "DoS/DoS"
]

# Map directory names to labels
label_map = {
    "Benign": "Benign",
    "BruteForce": "BruteForce",
    "DDOS": "DDoS",
    "DoS": "DoS"
}

all_dfs = []
stats = {}

print("Scanning for CSV files in specific folders...\n")

# Find CSV files only in target folders
for target_path in target_paths:
    if target_path.exists():
        for csv_file in sorted(target_path.glob("*.csv")):
            try:
                # Determine label from path
                label = "Unknown"
                for key, val in label_map.items():
                    if key in str(csv_file):
                        label = val
                        break
                
                # Read CSV
                df = pd.read_csv(csv_file)
                df['label'] = label  # Add label column
                
                all_dfs.append(df)
                stats[csv_file.name] = len(df)
                
                print(f"✓ {csv_file.name:<40} | {label:<10} | {len(df):>6} rows | Columns: {len(df.columns)}")
                
            except Exception as e:
                print(f"✗ {csv_file.name:<40} | Error: {str(e)[:40]}")
    else:
        print(f"⚠ {target_path} not found")

# Merge all DataFrames with union of all columns
if all_dfs:
    print("\n" + "="*80)
    print("Merging datasets...")
    
    # Get all unique columns
    all_columns = set()
    for df in all_dfs:
        all_columns.update(df.columns)
    
    print(f"Total unique columns: {len(all_columns)}")
    
    # Merge with all columns (fills missing with NaN)
    merged_df = pd.concat(all_dfs, axis=0, ignore_index=True, sort=True)
    
    print(f"✓ Merged shape: {merged_df.shape}")
    print(f"✓ Label distribution:\n{merged_df['label'].value_counts()}\n")
    
    # Save merged file
    output_path = csv_root / "merged_all_data.csv"
    merged_df.to_csv(output_path, index=False)
    print(f"✓ Saved to: {output_path}")
    print(f"  File size: {output_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("No CSV files found in the specified folders!")

Scanning for CSV files in specific folders...

✓ benign.csv                               | Benign     | 140319 rows | Columns: 32
✓ benign_2.csv                             | Benign     | 5132595 rows | Columns: 32
✓ benign_3.csv                             | Benign     | 853007 rows | Columns: 32
✓ benign_4.csv                             | Benign     | 7914616 rows | Columns: 32
✓ benign_5.csv                             | Benign     | 4948625 rows | Columns: 32
✓ benign_6.csv                             | Benign     | 3690697 rows | Columns: 32
✓ iperf_benign.csv                         | Benign     | 6529563 rows | Columns: 32
✓ ftp_hydra.csv                            | BruteForce | 1391246 rows | Columns: 32
✓ ftp_medusa.csv                           | BruteForce | 1488076 rows | Columns: 32
✓ ssh_hydra.csv                            | BruteForce | 12594947 rows | Columns: 32
✓ ssh_hydra2.csv                           | BruteForce | 5799943 rows | Columns: 32
✓ ssh_medusa.csv   

In [3]:
from pathlib import Path
import pandas as pd
from scapy.all import *
from scapy.error import Scapy_Exception

pcap_path = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\DDOS\DDOS\Network_Layer\ddos_icmp_hping3.pcap")


out_root = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output")
out_root.mkdir(parents=True, exist_ok=True)


def load_capture(path):
    path = str(path)

    with open(path, "rb") as f:
        magic = f.read(4)

    # PCAPNG magic number
    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)

    # Standard PCAP
    return PcapReader(path)


def packet_to_dict(pkt, pcap_path):

    row = {}

    row["pcap_file"] = pcap_path.name
    row["timestamp"] = float(pkt.time)
    row["packet_length"] = len(pkt)

    # ---------------- Ethernet ----------------

    if Ether in pkt:
        row["src_mac"] = pkt[Ether].src
        row["dst_mac"] = pkt[Ether].dst
        row["eth_type"] = pkt[Ether].type
    else:
        row["src_mac"] = None
        row["dst_mac"] = None
        row["eth_type"] = None

    # ---------------- IP ----------------

    if IP in pkt:

        ip = pkt[IP]

        row["src_ip"] = ip.src
        row["dst_ip"] = ip.dst
        row["ip_version"] = ip.version
        row["ttl"] = ip.ttl
        row["ip_length"] = ip.len
        row["tos"] = ip.tos
        row["id"] = ip.id
        row["flags_ip"] = int(ip.flags)
        row["fragment"] = ip.frag
        row["protocol"] = ip.proto

    else:

        row["src_ip"] = None
        row["dst_ip"] = None
        row["ip_version"] = None
        row["ttl"] = None
        row["ip_length"] = None
        row["tos"] = None
        row["id"] = None
        row["flags_ip"] = None
        row["fragment"] = None
        row["protocol"] = None

    # ---------------- TCP ----------------

    if TCP in pkt:

        tcp = pkt[TCP]

        row["src_port"] = tcp.sport
        row["dst_port"] = tcp.dport
        row["seq"] = tcp.seq
        row["ack"] = tcp.ack
        row["window"] = tcp.window

        row["syn"] = int(tcp.flags.S)
        row["ack_flag"] = int(tcp.flags.A)
        row["fin"] = int(tcp.flags.F)
        row["rst"] = int(tcp.flags.R)
        row["psh"] = int(tcp.flags.P)
        row["urg"] = int(tcp.flags.U)

    else:

        row["src_port"] = None
        row["dst_port"] = None
        row["seq"] = None
        row["ack"] = None
        row["window"] = None
        row["syn"] = 0
        row["ack_flag"] = 0
        row["fin"] = 0
        row["rst"] = 0
        row["psh"] = 0
        row["urg"] = 0

    # ---------------- UDP ----------------

    if UDP in pkt:

        udp = pkt[UDP]

        row["udp_length"] = udp.len

    else:

        row["udp_length"] = None

    # ---------------- ICMP ----------------

    if ICMP in pkt:

        icmp = pkt[ICMP]

        row["icmp_type"] = icmp.type
        row["icmp_code"] = icmp.code

    else:

        row["icmp_type"] = None
        row["icmp_code"] = None

    # ---------------- Payload ----------------

    row["payload_size"] = len(bytes(pkt.payload))

    return row


# ==========================================================
# Process the PCAP file
# ==========================================================

print(f"Processing: {pcap_path}")

try:

    packets = load_capture(pcap_path)

    rows = []

    for pkt in packets:
        rows.append(packet_to_dict(pkt, pcap_path))

    df = pd.DataFrame(rows)

    # Save CSV with the same filename as the PCAP
    csv_path = out_root / f"{pcap_path.stem}.csv"

    df.to_csv(csv_path, index=False)

    print(f"\nSuccessfully processed {len(df)} packets.")
    print(f"CSV saved to:\n{csv_path}")

except FileNotFoundError:
    print("Error: The specified PCAP file was not found.")

except Scapy_Exception as ex:
    print(f"Scapy error: {ex}")

except Exception as ex:
    print(f"Unexpected error: {ex}")

Processing: C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\DDOS\DDOS\Network_Layer\ddos_icmp_hping3.pcap

Successfully processed 409452 packets.
CSV saved to:
C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output\ddos_icmp_hping3.csv


In [6]:
from pathlib import Path
import pandas as pd
from scapy.all import *
from scapy.error import Scapy_Exception

pcap_path = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\DOS\DOS\Network_Layer\icmp_hping3.pcap")


out_root = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output")
out_root.mkdir(parents=True, exist_ok=True)


def load_capture(path):
    path = str(path)

    with open(path, "rb") as f:
        magic = f.read(4)

    # PCAPNG magic number
    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)

    # Standard PCAP
    return PcapReader(path)


def packet_to_dict(pkt, pcap_path):

    row = {}

    row["pcap_file"] = pcap_path.name
    row["timestamp"] = float(pkt.time)
    row["packet_length"] = len(pkt)

    # ---------------- Ethernet ----------------

    if Ether in pkt:
        row["src_mac"] = pkt[Ether].src
        row["dst_mac"] = pkt[Ether].dst
        row["eth_type"] = pkt[Ether].type
    else:
        row["src_mac"] = None
        row["dst_mac"] = None
        row["eth_type"] = None

    # ---------------- IP ----------------

    if IP in pkt:

        ip = pkt[IP]

        row["src_ip"] = ip.src
        row["dst_ip"] = ip.dst
        row["ip_version"] = ip.version
        row["ttl"] = ip.ttl
        row["ip_length"] = ip.len
        row["tos"] = ip.tos
        row["id"] = ip.id
        row["flags_ip"] = int(ip.flags)
        row["fragment"] = ip.frag
        row["protocol"] = ip.proto

    else:

        row["src_ip"] = None
        row["dst_ip"] = None
        row["ip_version"] = None
        row["ttl"] = None
        row["ip_length"] = None
        row["tos"] = None
        row["id"] = None
        row["flags_ip"] = None
        row["fragment"] = None
        row["protocol"] = None

    # ---------------- TCP ----------------

    if TCP in pkt:

        tcp = pkt[TCP]

        row["src_port"] = tcp.sport
        row["dst_port"] = tcp.dport
        row["seq"] = tcp.seq
        row["ack"] = tcp.ack
        row["window"] = tcp.window

        row["syn"] = int(tcp.flags.S)
        row["ack_flag"] = int(tcp.flags.A)
        row["fin"] = int(tcp.flags.F)
        row["rst"] = int(tcp.flags.R)
        row["psh"] = int(tcp.flags.P)
        row["urg"] = int(tcp.flags.U)

    else:

        row["src_port"] = None
        row["dst_port"] = None
        row["seq"] = None
        row["ack"] = None
        row["window"] = None
        row["syn"] = 0
        row["ack_flag"] = 0
        row["fin"] = 0
        row["rst"] = 0
        row["psh"] = 0
        row["urg"] = 0

    # ---------------- UDP ----------------

    if UDP in pkt:

        udp = pkt[UDP]

        row["udp_length"] = udp.len

    else:

        row["udp_length"] = None

    # ---------------- ICMP ----------------

    if ICMP in pkt:

        icmp = pkt[ICMP]

        row["icmp_type"] = icmp.type
        row["icmp_code"] = icmp.code

    else:

        row["icmp_type"] = None
        row["icmp_code"] = None

    # ---------------- Payload ----------------

    row["payload_size"] = len(bytes(pkt.payload))

    return row


# ==========================================================
# Process the PCAP file
# ==========================================================

print(f"Processing: {pcap_path}")

try:

    packets = load_capture(pcap_path)

    rows = []

    for pkt in packets:
        rows.append(packet_to_dict(pkt, pcap_path))

    df = pd.DataFrame(rows)

    # Save CSV with the same filename as the PCAP
    csv_path = out_root / f"{pcap_path.stem}.csv"

    df.to_csv(csv_path, index=False)

    print(f"\nSuccessfully processed {len(df)} packets.")
    print(f"CSV saved to:\n{csv_path}")

except FileNotFoundError:
    print("Error: The specified PCAP file was not found.")

except Scapy_Exception as ex:
    print(f"Scapy error: {ex}")

except Exception as ex:
    print(f"Unexpected error: {ex}")

Processing: C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\DOS\DOS\Network_Layer\icmp_hping3.pcap

Successfully processed 784241 packets.
CSV saved to:
C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output\icmp_hping3.csv


In [7]:
from pathlib import Path
import pandas as pd
from scapy.all import *
from scapy.error import Scapy_Exception

pcap_path = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\Benign\Benign\Network_Data\benign.pcap")


out_root = Path(r"C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output")
out_root.mkdir(parents=True, exist_ok=True)


def load_capture(path):
    path = str(path)

    with open(path, "rb") as f:
        magic = f.read(4)

    # PCAPNG magic number
    if magic == b"\x0a\x0d\x0d\x0a":
        return PcapNgReader(path)

    # Standard PCAP
    return PcapReader(path)


def packet_to_dict(pkt, pcap_path):

    row = {}

    row["pcap_file"] = pcap_path.name
    row["timestamp"] = float(pkt.time)
    row["packet_length"] = len(pkt)

    # ---------------- Ethernet ----------------

    if Ether in pkt:
        row["src_mac"] = pkt[Ether].src
        row["dst_mac"] = pkt[Ether].dst
        row["eth_type"] = pkt[Ether].type
    else:
        row["src_mac"] = None
        row["dst_mac"] = None
        row["eth_type"] = None

    # ---------------- IP ----------------

    if IP in pkt:

        ip = pkt[IP]

        row["src_ip"] = ip.src
        row["dst_ip"] = ip.dst
        row["ip_version"] = ip.version
        row["ttl"] = ip.ttl
        row["ip_length"] = ip.len
        row["tos"] = ip.tos
        row["id"] = ip.id
        row["flags_ip"] = int(ip.flags)
        row["fragment"] = ip.frag
        row["protocol"] = ip.proto

    else:

        row["src_ip"] = None
        row["dst_ip"] = None
        row["ip_version"] = None
        row["ttl"] = None
        row["ip_length"] = None
        row["tos"] = None
        row["id"] = None
        row["flags_ip"] = None
        row["fragment"] = None
        row["protocol"] = None

    # ---------------- TCP ----------------

    if TCP in pkt:

        tcp = pkt[TCP]

        row["src_port"] = tcp.sport
        row["dst_port"] = tcp.dport
        row["seq"] = tcp.seq
        row["ack"] = tcp.ack
        row["window"] = tcp.window

        row["syn"] = int(tcp.flags.S)
        row["ack_flag"] = int(tcp.flags.A)
        row["fin"] = int(tcp.flags.F)
        row["rst"] = int(tcp.flags.R)
        row["psh"] = int(tcp.flags.P)
        row["urg"] = int(tcp.flags.U)

    else:

        row["src_port"] = None
        row["dst_port"] = None
        row["seq"] = None
        row["ack"] = None
        row["window"] = None
        row["syn"] = 0
        row["ack_flag"] = 0
        row["fin"] = 0
        row["rst"] = 0
        row["psh"] = 0
        row["urg"] = 0

    # ---------------- UDP ----------------

    if UDP in pkt:

        udp = pkt[UDP]

        row["udp_length"] = udp.len

    else:

        row["udp_length"] = None

    # ---------------- ICMP ----------------

    if ICMP in pkt:

        icmp = pkt[ICMP]

        row["icmp_type"] = icmp.type
        row["icmp_code"] = icmp.code

    else:

        row["icmp_type"] = None
        row["icmp_code"] = None

    # ---------------- Payload ----------------

    row["payload_size"] = len(bytes(pkt.payload))

    return row


# ==========================================================
# Process the PCAP file
# ==========================================================

print(f"Processing: {pcap_path}")

try:

    packets = load_capture(pcap_path)

    rows = []

    for pkt in packets:
        rows.append(packet_to_dict(pkt, pcap_path))

    df = pd.DataFrame(rows)

    # Save CSV with the same filename as the PCAP
    csv_path = out_root / f"{pcap_path.stem}.csv"

    df.to_csv(csv_path, index=False)

    print(f"\nSuccessfully processed {len(df)} packets.")
    print(f"CSV saved to:\n{csv_path}")

except FileNotFoundError:
    print("Error: The specified PCAP file was not found.")

except Scapy_Exception as ex:
    print(f"Scapy error: {ex}")

except Exception as ex:
    print(f"Unexpected error: {ex}")

Processing: C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\Benign\Benign\Network_Data\benign.pcap

Successfully processed 140319 packets.
CSV saved to:
C:\Users\BLACKBOX\Desktop\Research Data\Iyenshi\Coding Assignment\new_csv_output\benign.csv
